# 07 — Tearsheets & Final Comparison
Pull everything from the results store: rank all runs, produce quantstats
tearsheets for the finalists, and break their performance down by regime.

### Results store ranking
All runs from chapters 03–05. Sort by Sharpe, pick best.


In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

### Re-run best + tearsheet
Quantstats HTML: Sharpe, Sortino, max DD, monthly rets, distributions.


In [ ]:
from lab.backtest import StrategyConfig, run_backtest
from lab.experiments import load_runs, load_trades
from lab.report import tearsheet, compare_runs, regime_breakdown

all_runs = load_runs()
print(f"{len(all_runs)} runs in the store, tags: {sorted(all_runs.tag.unique())}")
compare_runs().round(3).head(15)

### Regime breakdown
VIX quartile performance. Vol-dependent or robust?


In [ ]:
# Re-run the best config end-to-end and write its HTML tearsheet
best = all_runs.sort_values("sharpe_ratio", ascending=False).iloc[0]
print("best run:", best["name"], best.config_hash)
trades = load_trades(best.config_hash)
trades.tail()

### Final equity curve
Cumulative return over time. Smooth = good. Drawdowns = when/why?


In [ ]:
import json
cfg = StrategyConfig(
    name=best["name"], strategy=best["strategy"],
    params=json.loads(best["params_json"]),
    entry_filter=best["entry_filter"] or None,
    start=best["start"] or None, end=best["end"] or None,
)
result = run_backtest(cfg)
path = tearsheet(result)
print("tearsheet ->", path)

In [ ]:
regime_breakdown(result.trade_log, feature="vix_rank", bins=4).round(3)

In [ ]:
result.equity_curve.plot(title=f"{cfg.name} — final equity curve");

## Benchmark: are we beating the naive version of ourselves?
**CBOE PUT** is a systematic ATM put-write index — the public, investable
version of "just sell puts". Per the
[Wilshire/CBOE study](https://cdn.cboe.com/resources/spx/wilshire-options-based-benchmark-indexes-2019.pdf),
PUT historically delivered equity-like returns at ~2/3 the volatility.

A strategy only earns its complexity if it beats PUT (and SPX) after the extra
filters and management — check `excess_cagr_*`, `corr_*`, and `alpha_ann_*`.

In [ ]:
from lab.market_data import load_benchmark
from lab.metrics import compute_run_metrics
from lab.backtest import load_features

bench_metrics = {}
for sym in ("PUT", "BXM"):
    b = load_benchmark(sym)
    if b is not None:
        m = compute_run_metrics(result.trade_log, result.equity_curve,
                                features=load_features(), benchmark_close=b, benchmark_name=sym)
        bench_metrics.update({k: v for k, v in m.items() if sym in k})
pd.Series(bench_metrics).round(4)

In [ ]:
# Side-by-side growth of $1: strategy vs PUT vs SPX (common window)
from lab.metrics import daily_equity
from lab.market_data import load_benchmark, load_daily

eq = daily_equity(result.equity_curve)
frame = pd.DataFrame({"strategy": eq / eq.iloc[0]})
for sym in ("PUT", "SPX"):
    s = load_benchmark(sym) if sym == "PUT" else load_daily("SPX").set_index("quote_date")["close"]
    s = s.reindex(eq.index).ffill()
    frame[sym] = s / s.dropna().iloc[0]
frame.dropna().plot(title="Growth of $1: strategy vs CBOE PUT vs SPX");

## Publish: research memo + Sophie DB
The last step of a study: build the structured memo (hypothesis, method,
headline numbers, auto-caveats), render it to `results/reports/`, and push
runs + memo to the Sophie PostgreSQL (`option_research_run` /
`option_research_equity` / `option_research_evaluation`).

For the AI-written narrative, run the `/option-research-explain <tag>` skill
afterwards — it reads this memo, writes the four-section explanation
(tested / found / mechanism / falsification), and upserts the evaluation.

In [ ]:
from lab.explain import build_memo, render_memo, publish_study

STUDY_TAG = "baseline"      # any tag in the results store
memo = build_memo(STUDY_TAG, hypothesis="Signal-conditioned short puts beat unconditioned entries.")
print(render_memo(memo)[:1500])

In [ ]:
# Uncomment to push this study to the Sophie DB (runs + equity + memo):
# publish_study(STUDY_TAG, hypothesis=memo["hypothesis"])